# 20 — Chạy thử GPU có kiểm soát

Notebook này chỉ chạy smoke một fold và không mở tập test. Chạy E0 trước E1. Metrics tạo ra không phải kết quả nghiên cứu.

In [ ]:
from pathlib import Path
import os, subprocess, sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'configs' / 'experiments_v1.json').is_file():
    raise FileNotFoundError('Không tìm thấy workspace SleepTCN')
ENV = os.environ.copy()
ENV['PYTHONPATH'] = os.pathsep.join([str(ROOT), str(ROOT / 'src')])
EXPERIMENT = 'E2'  # đổi lần lượt: E0, E1, E2, E3
FOLD = 0
SEED = 42
print('Workspace:', ROOT)
print('Experiment:', EXPERIMENT, 'fold:', FOLD, 'seed:', SEED)

In [ ]:
subprocess.run([
    sys.executable, str(ROOT / 'scripts' / 'check_environment.py'),
    '--workspace', str(ROOT), '--require-gpu',
    '--output', str(ROOT / 'runs' / 'environment_check_gpu.json')
], env=ENV, check=True)

In [ ]:
subprocess.run([
    sys.executable, str(ROOT / 'scripts' / 'run_experiment.py'),
    '--workspace', str(ROOT), '--experiment', EXPERIMENT,
    '--fold', str(FOLD), '--seed', str(SEED),
    '--device', 'cuda', '--num-workers', '0', '--smoke'
], env=ENV, check=True)

In [ ]:
run_root = ROOT / 'runs' / 'smoke' / EXPERIMENT / f'fold_{FOLD:02d}' / f'seed_{SEED}'
subprocess.run([
    sys.executable, str(ROOT / 'scripts' / 'validate_run_artifacts.py'),
    '--workspace', str(ROOT), '--run-root', str(run_root),
    '--output', str(run_root / 'validation_report.json')
], env=ENV, check=True)